In [1]:
import pandas as pd
import numpy as np

# 1. Caricamento e Unione dei Dataset
# Sostituisci il percorso se i file non sono in data/raw/
df_train = pd.read_csv('data/raw/train.csv')
df_meal = pd.read_csv('data/raw/meal_info.csv')
df_center = pd.read_csv('data/raw/fulfilment_center_info.csv')

# Unione in un unico grande dataset
df = df_train.merge(df_meal, on='meal_id', how='left')
df = df.merge(df_center, on='center_id', how='left')

# Sanity Check: Assicuriamoci che non ci siano valori mancanti dopo l'unione
if df.isnull().sum().sum() == 0:
    print(f"Merge completato con successo. Dimensioni dataset: {df.shape}")
else:
    print(f"ATTENZIONE: Trovati valori nulli dopo il merge:\n{df.isnull().sum()}")

# ==========================================
# 2. Gestione Outliers e Trasformazione Dati
# ==========================================
# Il 99° percentile per questi dati si attesta a circa 1796 ordini.
limite_superiore = df['num_orders'].quantile(0.99)

# A. Tecnica del Capping per limitare i valori anomali
df['num_orders_capped'] = np.where(df['num_orders'] > limite_superiore, 
                                   limite_superiore, 
                                   df['num_orders'])

# B. Trasformazione Logaritmica (Super Consigliata per modelli di regressione)
# Stabilizza la varianza poiché la distribuzione è sbilanciata a sinistra (skewed)
df['log_num_orders'] = np.log1p(df['num_orders_capped'])

print(f"Outliers limitati al valore di: {limite_superiore:.0f} ordini")

# ==========================================
# 3. Mappatura delle Macro Categorie
# ==========================================
# Creiamo un dizionario di mappatura veloce per coprire il 100% delle 14 categorie presenti.
# Questo approccio è molto più veloce e affidabile della funzione if/else.
mappa_macro_categorie = {
    'Beverages': 'Bevande_e_Extra',
    'Extras': 'Bevande_e_Extra',
    'Soup': 'Pasti_Leggeri_e_Antipasti',
    'Salad': 'Pasti_Leggeri_e_Antipasti',
    'Starters': 'Pasti_Leggeri_e_Antipasti',
    'Other Snacks': 'Snack_e_Dolci',
    'Desert': 'Snack_e_Dolci',  # Gestisce anche l'errore di battitura "Desert" originale
    'Pizza': 'Fast_Food',
    'Pasta': 'Fast_Food',
    'Sandwich': 'Fast_Food',
    'Rice Bowl': 'Piatti_Principali',
    'Biryani': 'Piatti_Principali',
    'Fish': 'Piatti_Principali',
    'Seafood': 'Piatti_Principali'
}

# Applichiamo la mappatura utilizzando la funzione map() di Pandas
df['Macro_Categoria'] = df['category'].map(mappa_macro_categorie)

# Verifica: se ci fossero categorie sconosciute, verrebbero marcate come NaN (Non capiterà perché la mappa è completa)
if df['Macro_Categoria'].isnull().any():
    print("ATTENZIONE: Categorie non mappate trovate!")
    print(df[df['Macro_Categoria'].isnull()]['category'].unique())

# Visualizziamo un piccolo campione dei risultati finali
colonne_da_mostrare = ['meal_id', 'category', 'Macro_Categoria', 'num_orders', 'num_orders_capped', 'log_num_orders']
print("\nCampione dei dati elaborati:")
print(df[colonne_da_mostrare].head(10))

Merge completato con successo. Dimensioni dataset: (456548, 15)
Outliers limitati al valore di: 1796 ordini

Campione dei dati elaborati:
   meal_id   category  Macro_Categoria  num_orders  num_orders_capped  \
0     1885  Beverages  Bevande_e_Extra         177              177.0   
1     1993  Beverages  Bevande_e_Extra         270              270.0   
2     2539  Beverages  Bevande_e_Extra         189              189.0   
3     2139  Beverages  Bevande_e_Extra          54               54.0   
4     2631  Beverages  Bevande_e_Extra          40               40.0   
5     1248  Beverages  Bevande_e_Extra          28               28.0   
6     1778  Beverages  Bevande_e_Extra         190              190.0   
7     1062  Beverages  Bevande_e_Extra         391              391.0   
8     2707  Beverages  Bevande_e_Extra         472              472.0   
9     1207  Beverages  Bevande_e_Extra         676              676.0   

   log_num_orders  
0        5.181784  
1        5.602119 

In [2]:
# ==========================================
# 4. Feature Engineering (Creazione nuove variabili)
# ==========================================

# Ordiniamo i dati cronologicamente per ogni magazzino e per ogni pasto
df = df.sort_values(by=['center_id', 'meal_id', 'week'])

# Creiamo i Lags (Livello 1): "Quanti ordini ha fatto questo piatto in questo centro la settimana scorsa?"
# Lo facciamo sia sul dato originale (capped) sia sul logaritmo!
df['lag_1w'] = df.groupby(['center_id', 'meal_id'])['num_orders_capped'].shift(1)
df['lag_1w_log'] = df.groupby(['center_id', 'meal_id'])['log_num_orders'].shift(1)

# --- Lag aggiuntivi: 2 settimane fa e 4 settimane fa ---
df['lag_2w']     = df.groupby(['center_id', 'meal_id'])['num_orders_capped'].shift(2)
df['lag_2w_log'] = df.groupby(['center_id', 'meal_id'])['log_num_orders'].shift(2)
df['lag_4w']     = df.groupby(['center_id', 'meal_id'])['num_orders_capped'].shift(4)
df['lag_4w_log'] = df.groupby(['center_id', 'meal_id'])['log_num_orders'].shift(4)

# --- Rolling statistics (medie mobili e volatilità) ---
grp = df.groupby(['center_id', 'meal_id'])['num_orders_capped']
df['media_mobile_4w'] = grp.transform(lambda x: x.shift(1).rolling(4,  min_periods=1).mean())
df['media_mobile_8w'] = grp.transform(lambda x: x.shift(1).rolling(8,  min_periods=1).mean())
df['std_mobile_4w']   = grp.transform(lambda x: x.shift(1).rolling(4,  min_periods=1).std().fillna(0))

# --- Feature di calendario ---
df['mese_approx']     = ((df['week'] - 1) // 4) % 12 + 1   # mese approssimato dalla settimana
df['settimana_anno']  = ((df['week'] - 1) % 52) + 1         # settimana nell'anno (ciclica 1-52)
df['trimestre']       = ((df['mese_approx'] - 1) // 3) + 1  # trimestre 1-4
df['trimestre']       = df['trimestre'].astype('category')


# Variabili Economiche: lo sconto effettivo applicato
df['Sconto_Valore'] = df['base_price'] - df['checkout_price']

# NUOVO: Sconto Percentuale (molto più utile per i modelli predittivi)
# Usiamo np.where per evitare divisioni per zero se base_price fosse 0
df['Sconto_Percentuale'] = np.where(df['base_price'] > 0, 
                                    df['Sconto_Valore'] / df['base_price'], 
                                    0)
# ==========================================
# 5. Pulizia e Salvataggio
# ==========================================

# Rimuoviamo SOLO le righe dove manca il dato della settimana precedente (la prima settimana storica)
df_pulito = df.dropna(subset=['lag_1w']).copy()

# Salviamo il dataset finale pronto per il modello
import os
os.makedirs('data/processed', exist_ok=True)

# Salvataggio in CSV (Nota: togli l'index per evitare una colonna inutile di numeri)
df_pulito.to_csv('data/processed/dati_modello.csv', index=False)

print(f"Ingegneria dei dati completata. Passati da {len(df)} a {len(df_pulito)} righe.")
print("File 'dati_modello.csv' salvato in data/processed/!")

Ingegneria dei dati completata. Passati da 456548 a 452951 righe.
File 'dati_modello.csv' salvato in data/processed/!
